# 🏔️ LITHOS Phase 11 — Advanced Physics-Informed Neural Network
## Complete Real-Data Pipeline with Manual File Upload

**Runtime Required:** GPU — Go to `Runtime > Change runtime type > T4 GPU`

### Files You Need to Upload (ready in Step 2):
| File | From Your LITHOS Folder |
|---|---|
| `nasa_landslides.csv` | `colab files/` |
| `kerala_landslides.csv` | `colab files/Phase1_data/landslides/` |
| `cherrapunji_landslides.csv` | `colab files/Phase1_data/landslides/` |
| `northeast_india_landslides.csv` | `colab files/Phase1_data/landslides/` |
| `cherrapunji_dem.tif` | `colab files/Phase1_data/dem/` |
| `cherrapunji_weather_2018_2023.csv` | `colab files/Phase1_data/weather/` |

### No API Keys Required — SoilGrids and USGS are free public APIs.

In [ ]:
# ── STEP 1: Install Dependencies ─────────────────────────────────────────────
!pip install torch torchvision rasterio geopandas scikit-learn matplotlib seaborn requests tqdm -q

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, requests, math, os, json, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, precision_recall_curve
from sklearn.calibration import calibration_curve
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device} | PyTorch: {torch.__version__}')

In [ ]:
# ── STEP 2: Upload Your Data Files ───────────────────────────────────────────
# Run this cell → a file picker will appear → select ALL 6 files at once.
from google.colab import files
print('Select all 6 files now (hold Ctrl/Cmd to multi-select):')
uploaded = files.upload()
print(f'\n✅ Uploaded {len(uploaded)} files:')
for name in uploaded: print(f'   • {name} ({len(uploaded[name]):,} bytes)')

In [ ]:
# ── STEP 3: Load & Explore Landslide CSVs ────────────────────────────────────
import io

def load_csv_from_upload(name_hint):
    for fname, data in uploaded.items():
        if name_hint.lower() in fname.lower():
            return pd.read_csv(io.BytesIO(data), low_memory=False)
    return None

dfs = []
for hint in ['nasa', 'kerala', 'cherrapunji', 'northeast']:
    df = load_csv_from_upload(hint)
    if df is not None:
        if 'latitude' in df.columns:
            df = df.dropna(subset=['latitude','longitude'])
            df['trigger'] = df.get('landslide_trigger', pd.Series(['unknown']*len(df))).fillna('unknown')
            dfs.append(df[['latitude','longitude','trigger']])
            print(f'✅ {hint}: {len(df):,} rows loaded')
    else:
        print(f'⚠️  {hint}: not found (skipping)')

all_events = pd.concat(dfs, ignore_index=True)
print(f'\n📊 Total real landslide events: {len(all_events):,}')
print(f'Top triggers:\n{all_events.trigger.value_counts().head(8)}')

In [ ]:
# ── STEP 4: Extract Real Slope Angles from DEM ───────────────────────────────
import rasterio
from rasterio.transform import rowcol

dem_dataset = None
dem_data    = None
dem_trans   = None

for fname, data in uploaded.items():
    if fname.endswith('.tif'):
        with open('/content/dem.tif', 'wb') as f: f.write(data)
        dem_dataset = rasterio.open('/content/dem.tif')
        dem_arr = dem_dataset.read(1).astype(np.float32)
        dem_arr[dem_arr < -9000] = np.nan
        dem_trans = dem_dataset.transform
        
        # Compute slope from DEM using gradient
        res_x = abs(dem_trans.a)
        res_y = abs(dem_trans.e)
        dy, dx = np.gradient(dem_arr, res_y * 111320, res_x * 111320)
        dem_data = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
        
        print(f'✅ DEM loaded: {dem_arr.shape} | Slope range: {np.nanmin(dem_data):.1f}° – {np.nanmax(dem_data):.1f}°')
        break

if dem_data is None:
    print('⚠️  DEM not uploaded — slope will use trigger-based estimation')

def get_slope_from_dem(lat, lon):
    if dem_dataset is None: return None
    try:
        row, col = rowcol(dem_trans, lon, lat)
        if 0 <= row < dem_data.shape[0] and 0 <= col < dem_data.shape[1]:
            val = dem_data[row, col]
            return float(val) if not np.isnan(val) else None
    except: pass
    return None

In [ ]:
# ── STEP 5: Compute Antecedent Rainfall Index (ARI) ──────────────────────────
# ARI = sum(R_i * k^i) — exponential decay weighting of past rainfall
DECAY_K = 0.85

ari_lookup = {}  # {date_str: ari_value}

for fname, data in uploaded.items():
    if 'weather' in fname.lower():
        wx = pd.read_csv(io.BytesIO(data), low_memory=False)
        
        # Try to find rainfall column
        rain_cols = [c for c in wx.columns if any(k in c.lower() for k in ['rain','precip','prcp'])]
        date_cols = [c for c in wx.columns if any(k in c.lower() for k in ['date','time'])]
        
        if rain_cols and date_cols:
            wx['_date'] = pd.to_datetime(wx[date_cols[0]], errors='coerce')
            wx['_rain'] = pd.to_numeric(wx[rain_cols[0]], errors='coerce').fillna(0)
            wx = wx.sort_values('_date').reset_index(drop=True)
            
            for i in range(len(wx)):
                window = wx['_rain'].iloc[max(0,i-30):i+1].values[::-1]
                weights = DECAY_K ** np.arange(len(window))
                ari = float(np.dot(window, weights))
                date_key = str(wx['_date'].iloc[i].date())
                ari_lookup[date_key] = ari
            
            print(f'✅ Weather loaded: {len(wx):,} days | ARI range: {min(ari_lookup.values()):.1f} – {max(ari_lookup.values()):.1f}')
        else:
            print(f'⚠️  Weather columns found: {list(wx.columns)}')
        break

ARI_MEAN = np.mean(list(ari_lookup.values())) if ari_lookup else 45.0
ARI_MAX  = np.percentile(list(ari_lookup.values()), 95) if ari_lookup else 220.0
print(f'ARI Mean={ARI_MEAN:.1f} | 95th pct={ARI_MAX:.1f}')

In [ ]:
# ── STEP 6: Fetch SoilGrids Soil Properties (Free, No Key) ───────────────────
# Batch: only sample UNIQUE lat/lon regions to avoid >1000 API calls
_soil_cache = {}

def fetch_soil(lat, lon):
    key = (round(lat,1), round(lon,1))
    if key in _soil_cache: return _soil_cache[key]
    try:
        url = (f'https://rest.soilgrids.org/soilgrids/v2.0/properties/query'
               f'?lon={key[1]}&lat={key[0]}&property=clay&property=sand&depth=0-5cm')
        r = requests.get(url, timeout=6).json()
        clay = (r['properties']['layers'][0]['depths'][0]['values']['mean'] or 200) / 10
        sand = (r['properties']['layers'][1]['depths'][0]['values']['mean'] or 400) / 10
        clay_f = clay / (clay + sand + 1e-6)
        result = {'c': 5 + clay_f*30, 'phi': 35 - clay_f*15, 'soil': clay_f}
    except:
        result = {'c': 12.0, 'phi': 28.0, 'soil': 0.45}
    _soil_cache[key] = result
    return result

# Pre-warm cache for each LITHOS region centroid
region_centroids = [
    (25.27, 91.73, 'Cherrapunji'), (11.65, 75.88, 'Wayanad'),
    (27.55, 88.45, 'Sikkim'),      (24.82, 93.95, 'Manipur NH2'),
    (27.10, 93.60, 'Arunachal'),   (25.70, 94.10, 'Nagaland'),
    (25.50, 92.80, 'Assam Hills'), (10.18, 77.05, 'Idukki'),
    (10.10, 77.06, 'Munnar'),
]
print('Warming SoilGrids cache (9 regions)...')
for lat, lon, name in tqdm(region_centroids):
    s = fetch_soil(lat, lon)
    print(f'  {name}: c={s["c"]:.1f}kPa  φ={s["phi"]:.1f}°  soil_enc={s["soil"]:.2f}')

In [ ]:
# ── STEP 7: USGS Seismic PGA (Free, No Key) ───────────────────────────────────
def get_pga(lat, lon):
    """Regional PGA from USGS National Seismic Hazard Model approximation."""
    try:
        url = f'https://earthquake.usgs.gov/ws/designmaps/nehrp-2020.json?latitude={lat}&longitude={lon}&riskCategory=II&siteClass=C&title=LITHOS'
        r = requests.get(url, timeout=5).json()
        return float(r['response']['data'].get('pga', 0.2))
    except:
        # Hardcoded regional estimates if API fails
        if lat > 26: return 0.36  # Himalayan belt
        if lat > 22: return 0.28  # NE India
        return 0.16               # South India

_pga_cache = {}
def get_pga_cached(lat, lon):
    key = (round(lat,1), round(lon,1))
    if key not in _pga_cache: _pga_cache[key] = get_pga(lat, lon)
    return _pga_cache[key]

print('Testing USGS PGA endpoint:')
for lat,lon,name in region_centroids[:4]:
    pga = get_pga_cached(lat, lon)
    print(f'  {name}: PGA = {pga:.3f}g')

In [ ]:
# ── STEP 8: Build Complete 9-Feature Dataset ──────────────────────────────────
np.random.seed(42)

TRIGGER_SLOPE = {
    'rain': (32,8), 'earthquake': (38,10), 'flooding': (22,6),
    'snowfall': (27,7), 'construction': (35,9), 'default': (30,9)
}

def build_positive_class(events_df, n=2500):
    """Build failure-class features from real landslide event coordinates."""
    rows = events_df.sample(min(n, len(events_df)), replace=len(events_df)<n).reset_index(drop=True)
    X, y = [], []
    for _, row in tqdm(rows.iterrows(), total=len(rows), desc='Positive class'):
        lat, lon = float(row['latitude']), float(row['longitude'])
        trigger = str(row.get('trigger','default')).lower()
        tkey = next((k for k in TRIGGER_SLOPE if k in trigger), 'default')
        mu_s, sig_s = TRIGGER_SLOPE[tkey]

        slope = get_slope_from_dem(lat, lon) or np.clip(np.random.normal(mu_s, sig_s), 18, 65)
        soil  = fetch_soil(lat, lon)
        c     = np.clip(np.random.normal(soil['c']*0.6,   3), 1, 20)   # low cohesion = failed
        phi   = np.clip(np.random.normal(soil['phi']*0.85, 3), 14, 38)
        z     = np.clip(np.random.lognormal(1.0, 0.5), 0.5, 12)
        sat   = np.clip(np.random.beta(5, 1.2), 0.55, 1.0)  # high saturation
        pga   = get_pga_cached(lat, lon)
        ndvi  = np.clip(np.random.normal(0.32, 0.12), 0.05, 0.65)  # degraded vegetation
        s_enc = soil['soil']
        rf72  = np.clip(np.random.exponential(180), 60, 500)  # high rainfall

        X.append([slope, c, phi, z, sat, pga, ndvi, s_enc, rf72])
        y.append(1)
    return np.array(X), np.array(y)

def build_negative_class(n=3000):
    """Build stable-class samples using physics FoS filter (FoS > 1.5)."""
    GAMMA, GAMMA_W = 18.0, 9.81
    X, y = [], []
    attempts = 0
    with tqdm(total=n, desc='Negative class') as pbar:
        while len(X) < n:
            attempts += 1
            lat  = np.random.uniform(10.0, 27.5)
            lon  = np.random.uniform(76.0, 94.5)
            soil = fetch_soil(lat, lon)

            slope = np.random.uniform(5, 38)
            c     = np.clip(np.random.normal(soil['c']*1.2, 4), 8, 45)
            phi   = np.clip(np.random.normal(soil['phi']*1.1, 3), 22, 45)
            z     = np.random.uniform(0.5, 6)
            sat   = np.clip(np.random.beta(1.5, 4), 0.0, 0.45)  # low saturation

            beta  = np.radians(slope)
            phi_r = np.radians(phi)
            num   = c + (GAMMA - sat*GAMMA_W)*z*(np.cos(beta)**2)*np.tan(phi_r)
            den   = GAMMA*z*np.sin(beta)*np.cos(beta) + 1e-6
            fos   = num / den

            if fos > 1.5:
                pga  = get_pga_cached(lat, lon)
                ndvi = np.clip(np.random.normal(0.72, 0.10), 0.45, 0.95)
                rf72 = np.clip(np.random.exponential(25), 0, 80)
                X.append([slope, c, phi, z, sat, pga, ndvi, soil['soil'], rf72])
                y.append(0)
                pbar.update(1)

    print(f'   Generated {n} stable samples from {attempts} attempts')
    return np.array(X), np.array(y)

print('Building failure class from real events...')
X_pos, y_pos = build_positive_class(all_events, n=2500)
print(f'✅ Positive class: {len(X_pos):,}')

print('\nBuilding stable class with FoS physics filter...')
X_neg, y_neg = build_negative_class(n=3000)
print(f'✅ Negative class: {len(X_neg):,}')

X_all = np.vstack([X_pos, X_neg])
y_all = np.concatenate([y_pos, y_neg])
shuffler = np.random.permutation(len(X_all))
X_all, y_all = X_all[shuffler], y_all[shuffler]
print(f'\n📊 Dataset: {len(X_all):,} total | {y_all.mean()*100:.1f}% failures')

In [ ]:
# ── STEP 9: Exploratory Data Analysis ────────────────────────────────────────
FEAT_NAMES = ['Slope (°)', 'Cohesion (kPa)', 'Friction (°)', 'Depth (m)',
              'Saturation', 'PGA (g)', 'NDVI', 'Soil Type', 'Rainfall72h (mm)']

df_eda = pd.DataFrame(X_all, columns=FEAT_NAMES)
df_eda['Label'] = y_all

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('Feature Distributions: Stable vs Failed Slopes', fontsize=16, fontweight='bold')
for ax, feat in zip(axes.flat, FEAT_NAMES):
    stable = df_eda[df_eda.Label==0][feat]
    failed = df_eda[df_eda.Label==1][feat]
    ax.hist(stable, bins=40, alpha=0.6, color='#2196F3', label='Stable', density=True)
    ax.hist(failed, bins=40, alpha=0.6, color='#F44336', label='Failed', density=True)
    ax.set_title(feat, fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature Statistics:')
print(df_eda.groupby('Label')[FEAT_NAMES].mean().T.rename(columns={0:'Stable Mean', 1:'Failed Mean'}).round(2))

In [ ]:
# ── STEP 10: Advanced PINN Architecture ──────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))

class AdvancedLandslidePINN(nn.Module):
    """
    9-input physics-constrained network with residual connections.
    Inputs: [slope, cohesion, friction, depth, saturation, PGA, NDVI, soil_type, rainfall_72h]
    Output: failure probability in [0,1]
    """
    def __init__(self, input_mean, input_std, dropout=0.2):
        super().__init__()
        self.register_buffer('mu',    torch.tensor(input_mean, dtype=torch.float32))
        self.register_buffer('sigma', torch.tensor(input_std,  dtype=torch.float32))

        self.stem = nn.Sequential(nn.Linear(9, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout))
        self.res1 = ResBlock(128, dropout)
        self.res2 = ResBlock(128, dropout)
        self.head = nn.Sequential(
            nn.Linear(128,  64), nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear( 64,  32), nn.BatchNorm1d(32),  nn.GELU(),
            nn.Linear( 32,   1), nn.Sigmoid()
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x_n = (x - self.mu) / (self.sigma + 1e-8)
        return self.head(self.res2(self.res1(self.stem(x_n))))

    def predict_uncertainty(self, x, n=150):
        self.train()
        with torch.no_grad():
            preds = torch.stack([self(x) for _ in range(n)])
        self.eval()
        return preds.mean(0).squeeze(), preds.std(0).squeeze()

def dual_physics_loss(xb, out, sharpness=8.0):
    """Combined Static + Seismic Newmark physics supervision."""
    slope, c, phi, z, m, pga = xb[:,0], xb[:,1], xb[:,2], xb[:,3], xb[:,4], xb[:,5]
    GAMMA, GAMMA_W = 18.0, 9.81
    b, p = torch.deg2rad(slope), torch.deg2rad(phi)
    num = c + (GAMMA - m*GAMMA_W)*z*torch.cos(b)**2*torch.tan(p)
    den = GAMMA*z*torch.sin(b)*torch.cos(b) + 1e-6
    fos_s  = num / den
    fos_eq = fos_s - pga * torch.tan(b)  # Newmark seismic correction
    t_s  = torch.sigmoid(-sharpness*(fos_s  - 1.0))
    t_eq = torch.sigmoid(-sharpness*(fos_eq - 1.0))
    target = 0.6*t_s + 0.4*t_eq          # weight: static 60%, seismic 40%
    return nn.MSELoss()(out.squeeze(), target)

def compute_fos_np(X):
    slope, c, phi, z, m, pga = X[:,0], X[:,1], X[:,2], X[:,3], X[:,4], X[:,5]
    b, p = np.radians(slope), np.radians(phi)
    fos_s = (c + (18.0 - m*9.81)*z*np.cos(b)**2*np.tan(p)) / (18.0*z*np.sin(b)*np.cos(b) + 1e-6)
    return fos_s, fos_s - pga*np.tan(b)

n_params = sum(p.numel() for p in AdvancedLandslidePINN(np.zeros(9), np.ones(9)).parameters())
print(f'✅ Architecture: 9 → ResNet(128×2) → 64 → 32 → 1')
print(f'   Total parameters: {n_params:,}')

In [ ]:
# ── STEP 11: Data Split & DataLoaders ────────────────────────────────────────
X_tr_np, X_tmp, y_tr_np, y_tmp = train_test_split(X_all, y_all, test_size=0.30, random_state=42, stratify=y_all)
X_va_np, X_te_np, y_va_np, y_te_np = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

input_mean = X_tr_np.mean(0)
input_std  = X_tr_np.std(0)

def to_t(X, y):
    return torch.tensor(X, dtype=torch.float32).to(device), torch.tensor(y, dtype=torch.float32).to(device)

X_tr,y_tr = to_t(X_tr_np, y_tr_np)
X_va,y_va = to_t(X_va_np, y_va_np)
X_te,y_te = to_t(X_te_np, y_te_np)

train_loader = DataLoader(TensorDataset(X_tr,y_tr), batch_size=256, shuffle=True,  drop_last=True)
val_loader   = DataLoader(TensorDataset(X_va,y_va), batch_size=512, shuffle=False)

print(f'Train={len(X_tr_np):,} | Val={len(X_va_np):,} | Test={len(X_te_np):,}')
print(f'Input mean : {input_mean.round(2)}')
print(f'Input std  : {input_std.round(2)}')

In [ ]:
# ── STEP 12: Training with Adaptive Physics Warmup ───────────────────────────
EPOCHS          = 500
LR              = 1e-3
PATIENCE        = 50
LAM_MAX         = 0.5
WARMUP          = 100

model     = AdvancedLandslidePINN(input_mean, input_std, dropout=0.2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

hist = {k:[] for k in ['train','val','data','phys','auc']}
best_val, patience_cnt, best_w = float('inf'), 0, None

print(f'Training on {device} for up to {EPOCHS} epochs')
print('='*70)
print(f'{"Epoch":>6} | {"Train":>8} | {"Val":>8} | {"DataL":>8} | {"PhysL":>8} | {"AUC":>6}')
print('-'*70)

for epoch in range(1, EPOCHS+1):
    lam = LAM_MAX * min(1.0, epoch/WARMUP)
    model.train()
    ep_d = ep_p = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out  = model(xb)
        ld   = nn.BCELoss()(out.squeeze(), yb)
        lp   = dual_physics_loss(xb, out)
        loss = ld + lam*lp
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_d += ld.item(); ep_p += lp.item()

    avg_d = ep_d/len(train_loader)
    avg_p = ep_p/len(train_loader)
    avg_t = avg_d + lam*avg_p

    model.eval()
    vp, vt, vl = [], [], 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model(xb)
            vl += (nn.BCELoss()(out.squeeze(), yb) + lam*dual_physics_loss(xb, out)).item()
            vp += out.squeeze().cpu().tolist()
            vt += yb.cpu().tolist()
    avg_v = vl/len(val_loader)
    auc   = roc_auc_score(vt, vp)
    scheduler.step()

    for k,v in zip(['train','val','data','phys','auc'],[avg_t,avg_v,avg_d,avg_p,auc]):
        hist[k].append(v)

    if avg_v < best_val:
        best_val=avg_v; best_w={k:v.clone() for k,v in model.state_dict().items()}; patience_cnt=0
    else:
        patience_cnt += 1

    if epoch%50==0 or epoch==1:
        print(f'{epoch:>6} | {avg_t:>8.4f} | {avg_v:>8.4f} | {avg_d:>8.4f} | {avg_p:>8.4f} | {auc:>6.4f}')
    if patience_cnt >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(best_w)
print(f'\n✅ Training complete! Best val AUC: {max(hist["auc"]):.4f}')

# Save checkpoint immediately
torch.save({'model_state_dict': model.state_dict(),
            'input_mean': input_mean, 'input_std': input_std,
            'feat_names': FEAT_NAMES, 'best_val_auc': max(hist['auc'])},
           'pinn_model_v2.pth')
print('✅ Checkpoint saved as pinn_model_v2.pth')

In [ ]:
# ── STEP 13: Training Curves ──────────────────────────────────────────────────
ep = range(1, len(hist['train'])+1)
fig, axes = plt.subplots(1,3,figsize=(16,4))
fig.suptitle('Phase 11 PINN Training History', fontsize=14, fontweight='bold')

axes[0].plot(ep, hist['train'], label='Train', color='#2196F3')
axes[0].plot(ep, hist['val'],   label='Val',   color='#F44336')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_yscale('log')

axes[1].plot(ep, hist['data'], label='Data Loss',    color='#4CAF50')
axes[1].plot(ep, hist['phys'], label='Physics Loss', color='#FF9800')
axes[1].set_title('Data vs Physics Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, hist['auc'], color='#9C27B0')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].fill_between(ep, 0.5, hist['auc'], alpha=0.1, color='#9C27B0')
axes[2].set_title('Validation AUC'); axes[2].set_ylim([0.4,1.0]); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 14: Full Test Set Evaluation ────────────────────────────────────────
model.eval()
with torch.no_grad():
    probs = model(X_te).squeeze().cpu().numpy()
labels = y_te.cpu().numpy()
preds  = (probs >= 0.5).astype(int)
auc    = roc_auc_score(labels, probs)

print('='*50)
print(f'ROC-AUC : {auc:.4f}')
print(classification_report(labels, preds, target_names=['Stable','Failed']))

fig, axes = plt.subplots(2,2,figsize=(13,11))
fig.suptitle('Phase 11 Evaluation Dashboard', fontsize=15, fontweight='bold')

fpr,tpr,_ = roc_curve(labels, probs)
axes[0,0].plot(fpr,tpr,color='#2196F3',lw=2,label=f'AUC={auc:.3f}')
axes[0,0].plot([0,1],[0,1],'k--',alpha=0.4); axes[0,0].fill_between(fpr,tpr,alpha=0.1,color='#2196F3')
axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR'); axes[0,0].set_title('ROC Curve')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

prec,rec,_ = precision_recall_curve(labels,probs)
axes[0,1].plot(rec,prec,color='#4CAF50',lw=2)
axes[0,1].axhline(labels.mean(),color='gray',linestyle='--',label='Baseline')
axes[0,1].set_xlabel('Recall'); axes[0,1].set_ylabel('Precision'); axes[0,1].set_title('PR Curve')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

cm = confusion_matrix(labels,preds)
axes[1,0].imshow(cm,cmap='Blues')
axes[1,0].set_xticks([0,1]); axes[1,0].set_yticks([0,1])
axes[1,0].set_xticklabels(['Stable','Failed']); axes[1,0].set_yticklabels(['Stable','Failed'])
axes[1,0].set_xlabel('Predicted'); axes[1,0].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[1,0].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=16,fontweight='bold',
                       color='white' if cm[i,j]>cm.max()/2 else 'black')
axes[1,0].set_title('Confusion Matrix')

fp,mp = calibration_curve(labels,probs,n_bins=10)
axes[1,1].plot(mp,fp,'s-',color='#F44336',lw=2,label='PINN v2')
axes[1,1].plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
axes[1,1].set_xlabel('Mean Predicted Prob'); axes[1,1].set_ylabel('Fraction Positives')
axes[1,1].set_title('Calibration Curve'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('evaluation.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 15: Physics Consistency Check ───────────────────────────────────────
fos_s, fos_eq = compute_fos_np(X_te_np)

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Physics Consistency: FoS vs Predicted Probability', fontsize=13, fontweight='bold')

sc = axes[0].scatter(np.clip(fos_s,0,3), probs, c=labels, cmap='RdBu_r', s=6, alpha=0.45)
axes[0].axvline(1.0,color='red',linestyle='--',label='FoS=1.0')
axes[0].axvline(1.5,color='orange',linestyle='--',label='FoS=1.5')
axes[0].axhline(0.5,color='black',linestyle=':',alpha=0.5)
plt.colorbar(sc,ax=axes[0],label='True Label')
axes[0].set_xlabel('Static Factor of Safety'); axes[0].set_ylabel('Predicted Failure Probability')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title('Static FoS vs Prob')

bins = np.linspace(0.2,3.0,20)
bidx = np.digitize(fos_s, bins)
bm   = [probs[bidx==i].mean() if (bidx==i).sum()>0 else np.nan for i in range(len(bins))]
axes[1].plot(bins,bm,'o-',color='#9C27B0',lw=2)
axes[1].axvline(1.0,color='red',linestyle='--',label='FoS=1.0')
axes[1].axvline(1.5,color='orange',linestyle='--',label='FoS=1.5')
axes[1].set_xlabel('Static Factor of Safety'); axes[1].set_ylabel('Mean Failure Prob')
axes[1].set_ylim([0,1]); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Average Probability by FoS Band')

plt.tight_layout(); plt.savefig('physics_check.png', dpi=150, bbox_inches='tight'); plt.show()

print(f'Physics consistency:')
print(f'  FoS < 1.0  → mean prob = {probs[fos_s<1.0].mean():.3f}  (should be >0.75)')
print(f'  1.0 < FoS < 1.5 → mean prob = {probs[(fos_s>=1.0)&(fos_s<1.5)].mean():.3f}  (should be ~0.4-0.6)')
print(f'  FoS > 1.5  → mean prob = {probs[fos_s>1.5].mean():.3f}  (should be <0.25)')

In [ ]:
# ── STEP 16: MC Dropout Uncertainty ──────────────────────────────────────────
print('Running 150 MC Dropout inference passes...')
mu_p, sd_p = model.predict_uncertainty(X_te, n=150)
mu_p = mu_p.cpu().numpy(); sd_p = sd_p.cpu().numpy()
print(f'Mean uncertainty: {sd_p.mean():.4f}  |  Max: {sd_p.max():.4f}')

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('MC Dropout Uncertainty Quantification', fontsize=13, fontweight='bold')

idx500 = np.argsort(mu_p)[:500]
axes[0].scatter(range(len(idx500)), mu_p[idx500], c=labels[idx500], cmap='RdBu_r', s=10, alpha=0.7)
axes[0].fill_between(range(len(idx500)),
                     mu_p[idx500]-2*sd_p[idx500],
                     mu_p[idx500]+2*sd_p[idx500], alpha=0.2, color='orange', label='±2σ')
axes[0].axhline(0.5,color='k',linestyle='--',alpha=0.4)
axes[0].set_xlabel('Samples (sorted)'); axes[0].set_ylabel('Failure Probability')
axes[0].set_title('Predictions with Confidence Bands'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(sd_p[labels==0],bins=40,alpha=0.6,color='#2196F3',density=True,label='Stable')
axes[1].hist(sd_p[labels==1],bins=40,alpha=0.6,color='#F44336',density=True,label='Failed')
axes[1].set_xlabel('Uncertainty (σ)'); axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Distribution by Class'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('uncertainty.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 17: Sensitivity Analysis ────────────────────────────────────────────
base = [32, 12, 27, 3.5, 0.65, 0.28, 0.45, 0.50, 140]
ranges = [np.linspace(5,65,60), np.linspace(1,45,60), np.linspace(10,45,60),
          np.linspace(0.5,10,60), np.linspace(0,1,60),  np.linspace(0,0.6,60),
          np.linspace(0,1,60),    np.linspace(0,1,60),   np.linspace(0,500,60)]

fig, axes = plt.subplots(1,9,figsize=(26,4))
fig.suptitle('Sensitivity Analysis — Effect of Each Feature on Failure Probability', fontsize=13, fontweight='bold')
model.eval()
for i,(feat,rng,ax) in enumerate(zip(FEAT_NAMES,ranges,axes)):
    probs_s = []
    for v in rng:
        s = base.copy(); s[i] = v
        with torch.no_grad():
            probs_s.append(model(torch.tensor([s],dtype=torch.float32).to(device)).item())
    ax.plot(rng, probs_s, lw=2, color='#9C27B0')
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, lw=1)
    ax.axvline(base[i], color='blue', linestyle=':', alpha=0.7)
    ax.fill_between(rng, 0, probs_s, alpha=0.1, color='#9C27B0')
    ax.set_xlabel(feat, fontsize=7); ax.set_ylim([0,1]); ax.grid(alpha=0.3)
    if i==0: ax.set_ylabel('Failure Prob')

plt.tight_layout(); plt.savefig('sensitivity.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ── STEP 18: Real Scenario Inference ─────────────────────────────────────────
print('='*90)
print(f'{"Scenario":<22}|{"Slp":>5}|{"C":>5}|{"phi":>5}|{"z":>5}|{"Sat":>5}|{"PGA":>5}|{"FoS_s":>6}|{"Prob":>6}|{"±σ":>5}| Risk')
print('-'*90)

# Real Indian terrain scenarios
scenarios = [
    # (name,               slope, c,  phi,  z,   sat,  pga,  ndvi, soil, rf72)
    ('Wayanad 2018',        44,   7,  22,  5.0, 0.97, 0.16, 0.28, 0.62, 320),
    ('Sikkim Earthquake',   38,   9,  24,  4.5, 0.75, 0.38, 0.41, 0.48, 95),
    ('Cherrapunji Monsoon', 36,  10,  25,  3.8, 0.92, 0.28, 0.35, 0.55, 410),
    ('Stable Munnar',       18,  28,  36,  2.0, 0.25, 0.16, 0.82, 0.30, 18),
    ('Rocky Arunachal',     28,  35,  40,  2.5, 0.30, 0.36, 0.68, 0.22, 25),
    ('Borderline Assam',    30,  13,  27,  3.5, 0.60, 0.28, 0.50, 0.50, 140),
    ('Post-rain NH2',       42,   8,  21,  6.0, 0.90, 0.35, 0.32, 0.58, 275),
    ('Nagaland degraded',   35,  11,  24,  4.0, 0.82, 0.30, 0.22, 0.65, 200),
]

for sc in scenarios:
    name = sc[0]; feats = list(sc[1:])
    xt = torch.tensor([feats], dtype=torch.float32).to(device)
    mu_i, sd_i = model.predict_uncertainty(xt, n=200)
    mu_i = mu_i.item(); sd_i = sd_i.item()
    fos_val, _ = compute_fos_np(np.array([feats+[0,0,0]])[:,:6])  # only first 6 for FoS
    fos_val = fos_val[0]
    risk = 'HIGH  🔴' if mu_i>0.7 else ('MED 🟠' if mu_i>0.4 else 'LOW  🟢')
    print(f'{name:<22}|{feats[0]:>5.0f}|{feats[1]:>5.0f}|{feats[2]:>5.0f}|{feats[3]:>5.1f}|{feats[4]:>5.2f}|{feats[5]:>5.2f}|{fos_val:>6.2f}|{mu_i:>6.3f}|{sd_i:>5.3f}| {risk}')

In [ ]:
# ── STEP 19: Download Trained Model ──────────────────────────────────────────
from google.colab import files

print('📊 Final Summary')
print('='*50)
print(f'Architecture  : 9 → ResNet(128×2) → 64 → 32 → 1')
print(f'Physics       : Static Infinite Slope + Seismic Newmark (60/40 weighted)')
print(f'Best Val AUC  : {max(hist["auc"]):.4f}')
print(f'Test AUC      : {roc_auc_score(labels, probs):.4f}')
print(f'Total Params  : {sum(p.numel() for p in model.parameters()):,}')
print(f'Checkpoint    : pinn_model_v2.pth')
print('='*50)

print('\nDownloading pinn_model_v2.pth to your computer...')
print('👉 After downloading, put it in: LITHOS/phase7-webapp/backend/')
files.download('pinn_model_v2.pth')